# Comprehensive EDA of Brazilian Human Rights Hotline Complaints

## Step 4: Univariate Analysis

This notebook describes each variable independently after the dataset has been cleaned and validated. It focuses on the marginal distributions of complaint, location, victim, suspect, relationship, and violation attributes using interactive Plotly visualizations.

### Analytical objectives

1. Define the correct counting unit before producing frequencies.
2. Profile coverage and cardinality of analysis variables.
3. Describe complaint intake and emergency classification.
4. Analyze violation, vulnerable-group, and contextual distributions.
5. Describe victim and suspect characteristics independently.
6. Examine geographic coverage without interpreting counts as population rates.
7. Analyze the distribution of the number of victims.
8. Identify rare categories and dominant categories that affect later bivariate analysis.
9. Export tables, figures, and a univariate findings register.

> **Interpretive constraint:** hotline records measure reported complaints recorded by the system. They do not directly estimate the prevalence or incidence of human rights violations in the Brazilian population.


## 4.1 Imports, paths, and analytical settings

The notebook first tries to load the cleaned Parquet file produced in Step 3 and falls back to CSV. `COUNTING_MODE` controls whether distributions represent rows or unique complaints.

- `record`: each row contributes once. Use this for row-level attributes such as `violacao` when the file grain is one violation per row.
- `complaint`: one observation per unique `hash`. Use this for complaint-level attributes after confirming that values are stable within each hash.

The default is conservative: `record`. Change it only after resolving the dataset-grain analysis from Steps 1 to 3.


In [51]:
from pathlib import Path
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 150)
pd.set_option("display.max_colwidth", 160)

PARQUET_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.parquet")
CSV_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.csv")
OUTPUT_DIR = Path("outputs/step_04_univariate_analysis")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

COUNTING_MODE = "record"  # record or complaint
IDENTIFIER = "hash"
TOP_N = 20
RARE_CATEGORY_MIN_COUNT = 10
RARE_CATEGORY_MIN_PERCENT = 0.5
INCLUDE_MISSING_IN_CHARTS = True
EXPORT_FIGURES_HTML = True

PLOTLY_TEMPLATE = "plotly_white"
COLOR_SEQUENCE = px.colors.qualitative.Safe


## 4.2 Load the cleaned analytical dataset


In [52]:
def load_clean_dataset(parquet_path, csv_path):
    if parquet_path.exists():
        data = pd.read_parquet(parquet_path)
        source = parquet_path
    elif csv_path.exists():
        data = pd.read_csv(csv_path, low_memory=False)
        source = csv_path
    else:
        raise FileNotFoundError(
            "Cleaned dataset not found. Run Step 3 or update PARQUET_PATH/CSV_PATH."
        )
    return data, source


df, DATA_SOURCE = load_clean_dataset(PARQUET_PATH, CSV_PATH)
print("Source:", DATA_SOURCE.resolve())
print(f"Rows: {len(df):,} | Columns: {df.shape[1]:,}")
display(df.head(3))


Source: C:\Projetos\eda-disque100-completo\eda\outputs\step_03_data_cleaning\disque100_clean.parquet
Rows: 8,401 | Columns: 66


,hash,data_de_cadastro,canal_de_atendimento,denuncia_emergencial,denunciante,cenario_da_violacao,pais,uf,municipio,frequencia,inicio_das_violacoes,sl_quantidade_vitimas,grupo_vulneravel,motivacao,relacao_vitima_suspeito,sl_vitima_cadastro,genero_da_vitima,orientacao_sexual_da_vitima,faixa_etaria_da_vitima,nacionalidade_da_vitima,sl_vitima_naturalidade,sl_vitima_naturalizado_uf,sl_vitima_naturalizado_municipio,deficiencia_da_vitima,doenca_rara_da_vitima,deficiencia_relacionada_a_doenca_rara,vitima_preso_a,pais_da_vitima,uf_da_vitima,municipio_da_vitima,profissao_da_vitima,grau_de_instrucao_da_vitima,religiao_da_vitima,raca_cor_da_vitima,etnia_da_vitima,faixa_de_renda_da_vitima,natureza_juridica_do_suspeito,genero_do_suspeito,orientacao_sexual_do_suspeito,faixa_etaria_do_suspeito,suspeito_nacionalidade,sl_suspeito_naturalidade,sl_suspeito_naturalizado_uf,sl_suspeito_naturalizado_municipio,deficiencia_do_suspeito,doenca_rara_do_suspeito,deficiencia_relacionada_a_doenca_rara_suspeito,suspeito_preso,pais_do_suspeito,uf_do_suspeito,sl_suspeito_municipio,profissao_do_suspeito,grau_de_instrucao_do_suspeito,religiao_do_suspeito,raca_cor_do_suspeito,suspeito_etnia,faixa_de_renda_do_suspeito,vinculo_orgao_pj_do_suspeito,sl_suspeito_ramo,suspeito_especificacao_etnia,vitima_especificacao_etnia,violacao,ano_cadastro,mes_cadastro,ano_mes_cadastro,dia_semana_cadastro
0,79C48E0D422D3AB1A306F8D7E498B8BE,2021-01-08 23:11:00,TELEFÔNICO,NÃO,ANÔNIMO,SERVIÇO DE ABRIGAMENTO,BRASIL,SP,SÃO BERNARDO DO CAMPO,DIARIAMENTE,HÁ MAIS DE CINCO ANOS,10,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,PARA OBTENÇÃO DE BENEFÍCIO FINANCEIRO/ GANÂNCIA,OUTROS,PESSOA FÍSICA,FEMININO,N/D,<NA>,N/D,<NA>,<NA>,<NA>,N/D,NÃO TEM DOENÇA RARA,<NA>,N/D,BRASIL,SP,SÃO BERNARDO DO CAMPO,N/D,N/D,N/D,N/D,N/D,N/D,PESSOA FÍSICA,MASCULINO,N/D,60 A 64 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO,NÃO,BRASIL,SP,SÃO BERNARDO DO CAMPO,N/D,N/D,N/D,N/D,N/D,N/D,NaN,N/D,N/D,N/D,INTEGRIDADE>PSÍQUICA>CALÚNIA,2021,1,2021-01,Friday
1,0B9E67B28384351A4006C49BA3FF66B7,2021-01-14 07:11:00,TELEFÔNICO,NÃO,ANÔNIMO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,BRASIL,SP,SÃO PAULO,DIARIAMENTE,HÁ MAIS DE SEIS MESES,10,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,DA COABITAÇÃO/ CONVIVÊNCIA FAMILIAR/ RELAÇÃO AFETIVA,FAMILIARES,PESSOA FÍSICA,FEMININO,N/D,02 A 04 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,BRANCA,N/D,ATÉ 1 SM,PESSOA FÍSICA,FEMININO,N/D,35 A 39 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO,NÃO,BRASIL,SP,SÃO PAULO,N/D,ENSINO FUNDAMENTAL INCOMPLETO,N/D,BRANCA,N/D,ATÉ 1 SM,NaN,N/D,N/D,N/D,INTEGRIDADE>PSÍQUICA>AMEAÇA OU COAÇÃO,2021,1,2021-01,Thursday
2,6FA1ED73EC4C3F385683C9FC03A3C062,2021-01-12 07:34:00,TELEFÔNICO,NÃO,ANÔNIMO,CASA DA VÍTIMA,BRASIL,SP,SÃO PAULO,DIARIAMENTE,HÁ MAIS DE SEIS MESES,20,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,DA COABITAÇÃO/ CONVIVÊNCIA FAMILIAR/ RELAÇÃO AFETIVA,PAI,PESSOA FÍSICA,FEMININO,N/D,02 A 04 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,<NA>,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,BRANCA,N/D,N/D,PESSOA FÍSICA,MASCULINO,N/D,45 A 49 ANOS,BRASIL,NATO,SP,SÃO PAULO,N/D,NÃO TEM DOENÇA RARA,NÃO SABE INFORMAR,N/D,BRASIL,SP,SÃO PAULO,N/D,N/D,N/D,PRETA,N/D,N/D,NaN,N/D,N/D,N/D,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESSOAS - INTERNACIONAL,2021,1,2021-01,Tuesday


## 4.3 Verify readiness for analysis

The notebook stops if essential assumptions are not met. Review warnings before interpreting the charts.


In [53]:
readiness_rows = []
readiness_rows.append({
    "check": "dataset_not_empty", "status": "PASS" if len(df) > 0 else "FAIL",
    "value": len(df)
})
readiness_rows.append({
    "check": "columns_unique", "status": "PASS" if df.columns.is_unique else "FAIL",
    "value": int(df.columns.duplicated().sum())
})
readiness_rows.append({
    "check": "identifier_available", "status": "PASS" if IDENTIFIER in df.columns else "REVIEW",
    "value": IDENTIFIER in df.columns
})
if COUNTING_MODE not in {"record", "complaint"}:
    raise ValueError("COUNTING_MODE must be 'record' or 'complaint'.")
if COUNTING_MODE == "complaint" and IDENTIFIER not in df.columns:
    raise ValueError("Complaint mode requires the identifier column.")

readiness = pd.DataFrame(readiness_rows)
display(readiness)
if (readiness["status"] == "FAIL").any():
    raise ValueError("The analytical dataset failed a readiness check.")


,check,status,value
0,dataset_not_empty,PASS,8401
1,columns_unique,PASS,0
2,identifier_available,PASS,True


## 4.4 Define the analytical population and denominators

Complaint mode retains one row per `hash`. Before using it, inspect whether each requested complaint-level variable has more than one non-null value within a hash. Conflicting variables should not be summarized from an arbitrary first row.


In [54]:
PRIORITY_VARIABLES = [
    "canal_de_atendimento", "denuncia_emergencial", "denunciante",
    "cenario_da_violacao", "frequencia", "inicio_das_violacoes",
    "grupo_vulneravel", "motivacao", "relacao_vitima_suspeito", "violacao",
    "pais", "uf", "municipio", "genero_da_vitima",
    "orientacao_sexual_da_vitima", "faixa_etaria_da_vitima",
    "nacionalidade_da_vitima", "deficiencia_da_vitima",
    "doenca_rara_da_vitima", "vitima_preso_a", "raca_cor_da_vitima",
    "genero_do_suspeito", "orientacao_sexual_do_suspeito",
    "faixa_etaria_do_suspeito", "suspeito_nacionalidade",
    "deficiencia_do_suspeito", "doenca_rara_do_suspeito",
    "suspeito_preso", "raca_cor_do_suspeito", "natureza_juridica_do_suspeito"
]

present_priority = [c for c in PRIORITY_VARIABLES if c in df.columns]
conflict_rows = []
if IDENTIFIER in df.columns:
    valid_id = df[df[IDENTIFIER].notna()]
    for column in present_priority:
        within_id_unique = valid_id.groupby(IDENTIFIER)[column].nunique(dropna=True)
        conflict_rows.append({
            "column": column,
            "hashes_with_multiple_non_null_values": int((within_id_unique > 1).sum()),
            "maximum_values_within_hash": int(within_id_unique.max()) if len(within_id_unique) else 0
        })
identifier_conflicts = pd.DataFrame(conflict_rows).sort_values(
    "hashes_with_multiple_non_null_values", ascending=False
) if conflict_rows else pd.DataFrame()
display(identifier_conflicts)

if COUNTING_MODE == "complaint":
    df_analysis = df.dropna(subset=[IDENTIFIER]).drop_duplicates(subset=[IDENTIFIER], keep="first").copy()
    unit_label = "complaints"
else:
    df_analysis = df.copy()
    unit_label = "records"

population_summary = pd.DataFrame({
    "metric": ["Counting mode", "Analytical observations", "Source rows", "Unique non-null hashes"],
    "value": [COUNTING_MODE, len(df_analysis), len(df), df[IDENTIFIER].nunique(dropna=True) if IDENTIFIER in df else np.nan]
})
display(population_summary)


,column,hashes_with_multiple_non_null_values,maximum_values_within_hash
9,violacao,1130,48
7,motivacao,500,17
25,deficiencia_do_suspeito,138,2
17,deficiencia_da_vitima,116,3
6,grupo_vulneravel,53,2
15,faixa_etaria_da_vitima,7,2
21,genero_do_suspeito,4,2
3,cenario_da_violacao,4,2
23,faixa_etaria_do_suspeito,3,2
28,raca_cor_do_suspeito,1,2


,metric,value
0,Counting mode,record
1,Analytical observations,8401
2,Source rows,8401
3,Unique non-null hashes,1263


## 4.5 Reusable frequency and Plotly functions

Percentages use all analytical observations as the denominator when missing values are included. The table also reports valid percentages, which use only non-missing observations. This distinction is essential for sparse administrative variables.


In [55]:
def frequency_table(data, column, include_missing=True):
    series = data[column].astype("string")
    missing = series.isna()
    display_series = series.fillna("<MISSING>") if include_missing else series.dropna()
    counts = display_series.value_counts(dropna=False)
    table = counts.rename("count").to_frame().reset_index(names="category")
    table["percent_total"] = (table["count"] / len(data) * 100).round(2)
    valid_n = int((~missing).sum())
    table["percent_valid"] = np.where(
        table["category"].eq("<MISSING>"), np.nan,
        (table["count"] / valid_n * 100).round(2) if valid_n else np.nan
    )
    table["cumulative_percent_total"] = table["percent_total"].cumsum().round(2)
    return table


def category_metrics(data, column):
    series = data[column]
    valid = series.dropna()
    freq = valid.value_counts()
    return {
        "column": column,
        "dtype": str(series.dtype),
        "total_n": len(series),
        "valid_n": int(series.notna().sum()),
        "missing_n": int(series.isna().sum()),
        "missing_pct": round(series.isna().mean()*100, 2),
        "unique_valid": int(valid.nunique()),
        "mode": freq.index[0] if len(freq) else pd.NA,
        "mode_n": int(freq.iloc[0]) if len(freq) else 0,
        "mode_pct_valid": round(freq.iloc[0]/len(valid)*100, 2) if len(valid) else np.nan,
    }


def plot_frequency(data, column, title=None, top_n=20, include_missing=True, other_label="OTHER"):
    table = frequency_table(data, column, include_missing)
    if len(table) > top_n:
        top = table.head(top_n).copy()
        remainder = table.iloc[top_n:]
        other = pd.DataFrame({
            "category": [other_label],
            "count": [int(remainder["count"].sum())],
            "percent_total": [round(remainder["percent_total"].sum(), 2)],
            "percent_valid": [np.nan],
            "cumulative_percent_total": [100.0]
        })
        chart_table = pd.concat([top, other], ignore_index=True)
    else:
        chart_table = table.copy()
    chart_table = chart_table.sort_values("count", ascending=True)
    fig = px.bar(
        chart_table, x="count", y="category", orientation="h", text="count",
        hover_data=["percent_total", "percent_valid"],
        title=title or f"Distribution of {column}",
        labels={"count": unit_label.title(), "category": column},
        color="count", color_continuous_scale="Blues",
        height=max(450, 28*len(chart_table)+120)
    )
    fig.update_layout(template=PLOTLY_TEMPLATE, coloraxis_showscale=False)
    fig.update_traces(textposition="outside", cliponaxis=False)
    return fig, table


def save_figure(fig, filename):
    if EXPORT_FIGURES_HTML:
        fig.write_html(FIGURE_DIR / filename, include_plotlyjs=True)


## 4.6 Variable-level univariate inventory

This table identifies missingness, cardinality, mode, and concentration. A high modal percentage may reflect a genuinely dominant category, a default form value, or a collection-process artifact.


In [56]:
analysis_variables = [c for c in PRIORITY_VARIABLES if c in df_analysis.columns]
univariate_inventory = pd.DataFrame([category_metrics(df_analysis, c) for c in analysis_variables])
univariate_inventory["mode"] = univariate_inventory["mode"].astype("string")
univariate_inventory = univariate_inventory.sort_values(["missing_pct", "unique_valid"], ascending=[False, False])
display(univariate_inventory)
univariate_inventory.to_csv(TABLE_DIR / "univariate_inventory.csv", index=False, encoding="utf-8-sig")


,column,dtype,total_n,valid_n,missing_n,missing_pct,unique_valid,mode,mode_n,mode_pct_valid
25,deficiencia_do_suspeito,string,8401,1921,6480,77.13,6,NÃO,971,50.55
17,deficiencia_da_vitima,string,8401,1956,6445,76.72,8,NÃO,966,49.39
7,motivacao,string,8401,2936,5465,65.05,87,OUTRO(S) ELEMENTO(S),773,26.33
28,raca_cor_do_suspeito,string,8401,3729,4672,55.61,5,BRANCA,1715,45.99
27,suspeito_preso,string,8401,4037,4364,51.95,8,NÃO,3121,77.31
23,faixa_etaria_do_suspeito,string,8401,4567,3834,45.64,18,30 A 34 ANOS,716,15.68
20,raca_cor_da_vitima,string,8401,4981,3420,40.71,6,BRANCA,2014,40.43
19,vitima_preso_a,string,8401,5395,3006,35.78,2,NÃO,4655,86.28
14,orientacao_sexual_da_vitima,string,8401,5639,2762,32.88,9,NÃO,4313,76.49
21,genero_do_suspeito,string,8401,6520,1881,22.39,5,MASCULINO,3127,47.96


## 4.7 Complaint intake and context

These variables describe how the complaint entered the system and the reported setting or recurrence. They characterize recorded complaints, not the underlying prevalence of violations.


In [57]:
COMPLAINT_VARIABLES = [
    "canal_de_atendimento", "denuncia_emergencial", "denunciante",
    "cenario_da_violacao", "frequencia", "inicio_das_violacoes"
]
complaint_tables = {}
for column in [c for c in COMPLAINT_VARIABLES if c in df_analysis.columns]:
    fig, table = plot_frequency(
        df_analysis, column,
        title=f"Complaint profile: {column}", top_n=TOP_N,
        include_missing=INCLUDE_MISSING_IN_CHARTS
    )
    fig.show()
    save_figure(fig, f"complaint_{column}.html")
    table.to_csv(TABLE_DIR / f"frequency_{column}.csv", index=False, encoding="utf-8-sig")
    complaint_tables[column] = table


## 4.8 Human rights violations and vulnerable groups

`violacao` may be hierarchical or repeated within a complaint. In record mode, the chart describes violation records. In complaint mode, it should be used only if the conflict table confirms that the variable is stable within `hash` or after a documented multi-label aggregation.


In [58]:
OUTCOME_VARIABLES = ["violacao", "grupo_vulneravel", "motivacao"]
outcome_tables = {}
for column in [c for c in OUTCOME_VARIABLES if c in df_analysis.columns]:
    fig, table = plot_frequency(
        df_analysis, column,
        title=f"Recorded distribution: {column}", top_n=TOP_N,
        include_missing=INCLUDE_MISSING_IN_CHARTS
    )
    fig.show()
    save_figure(fig, f"outcome_{column}.html")
    table.to_csv(TABLE_DIR / f"frequency_{column}.csv", index=False, encoding="utf-8-sig")
    outcome_tables[column] = table


## 4.9 Victim profile

Victim fields can be missing because information was unknown, not collected, inapplicable, or because the row does not represent a unique victim. Results should be reported with both total and valid denominators.


In [59]:
VICTIM_VARIABLES = [
    "genero_da_vitima", "orientacao_sexual_da_vitima", "faixa_etaria_da_vitima",
    "nacionalidade_da_vitima", "deficiencia_da_vitima", "doenca_rara_da_vitima",
    "vitima_preso_a", "profissao_da_vitima", "grau_de_instrucao_da_vitima",
    "religiao_da_vitima", "raca_cor_da_vitima", "etnia_da_vitima",
    "faixa_de_renda_da_vitima"
]
victim_tables = {}
for column in [c for c in VICTIM_VARIABLES if c in df_analysis.columns]:
    fig, table = plot_frequency(
        df_analysis, column,
        title=f"Victim profile: {column}", top_n=TOP_N,
        include_missing=INCLUDE_MISSING_IN_CHARTS
    )
    fig.show()
    save_figure(fig, f"victim_{column}.html")
    table.to_csv(TABLE_DIR / f"frequency_{column}.csv", index=False, encoding="utf-8-sig")
    victim_tables[column] = table


## 4.10 Suspect profile

These distributions describe information recorded about alleged suspects. They should not be interpreted as population characteristics or as findings of legal responsibility.


In [60]:
SUSPECT_VARIABLES = [
    "natureza_juridica_do_suspeito", "genero_do_suspeito",
    "orientacao_sexual_do_suspeito", "faixa_etaria_do_suspeito",
    "suspeito_nacionalidade", "deficiencia_do_suspeito",
    "doenca_rara_do_suspeito", "suspeito_preso", "profissao_do_suspeito",
    "grau_de_instrucao_do_suspeito", "religiao_do_suspeito",
    "raca_cor_do_suspeito", "suspeito_etnia", "faixa_de_renda_do_suspeito",
    "sl_suspeito_ramo"
]
suspect_tables = {}
for column in [c for c in SUSPECT_VARIABLES if c in df_analysis.columns]:
    fig, table = plot_frequency(
        df_analysis, column,
        title=f"Suspect profile: {column}", top_n=TOP_N,
        include_missing=INCLUDE_MISSING_IN_CHARTS
    )
    fig.show()
    save_figure(fig, f"suspect_{column}.html")
    table.to_csv(TABLE_DIR / f"frequency_{column}.csv", index=False, encoding="utf-8-sig")
    suspect_tables[column] = table


## 4.11 Victim-suspect relationship

This variable is often central to prevention and service-delivery analysis because it distinguishes family, intimate-partner, institutional, community, and other contexts.


In [61]:
if "relacao_vitima_suspeito" in df_analysis.columns:
    relationship_fig, relationship_table = plot_frequency(
        df_analysis, "relacao_vitima_suspeito",
        title="Reported relationship between victim and suspect",
        top_n=TOP_N, include_missing=INCLUDE_MISSING_IN_CHARTS
    )
    relationship_fig.show()
    save_figure(relationship_fig, "relationship_victim_suspect.html")
    relationship_table.to_csv(TABLE_DIR / "frequency_relacao_vitima_suspeito.csv", index=False, encoding="utf-8-sig")


## 4.12 Geographic distributions

Raw complaint counts by UF or municipality must not be interpreted as risk rates. Population size, internet and telephone access, awareness of the hotline, institutional capacity, and reporting behavior all affect recorded volumes. Population-standardized rates require an external denominator for the same reference period.


In [62]:
GEOGRAPHIC_VARIABLES = [
    "pais", "uf", "municipio", "pais_da_vitima", "uf_da_vitima",
    "municipio_da_vitima", "pais_do_suspeito", "uf_do_suspeito",
    "sl_suspeito_municipio"
]
geographic_tables = {}
for column in [c for c in GEOGRAPHIC_VARIABLES if c in df_analysis.columns]:
    top_n = 30 if "municipio" in column else TOP_N
    fig, table = plot_frequency(
        df_analysis, column,
        title=f"Geographic distribution: {column}", top_n=top_n,
        include_missing=INCLUDE_MISSING_IN_CHARTS
    )
    fig.show()
    save_figure(fig, f"geography_{column}.html")
    table.to_csv(TABLE_DIR / f"frequency_{column}.csv", index=False, encoding="utf-8-sig")
    geographic_tables[column] = table


## 4.13 Number of victims per record

Because collective complaints can create a long right tail, the analysis reports robust statistics and shows both the full range and a 99th-percentile zoom. High values are review candidates, not automatic errors.


In [63]:
if "sl_quantidade_vitimas" in df_analysis.columns:
    victim_quantity = pd.to_numeric(df_analysis["sl_quantidade_vitimas"], errors="coerce")
    quantity_summary = victim_quantity.describe(
        percentiles=[.01, .05, .25, .5, .75, .9, .95, .99]
    ).rename("value").to_frame()
    q1, q3 = victim_quantity.quantile([.25, .75])
    iqr = q3 - q1
    quantity_diagnostics = pd.DataFrame({
        "metric": ["IQR", "IQR upper review fence", "Records above fence", "99th percentile"],
        "value": [iqr, q3 + 1.5*iqr, int((victim_quantity > q3 + 1.5*iqr).sum()), victim_quantity.quantile(.99)]
    })
    display(quantity_summary)
    display(quantity_diagnostics)

    quantity_df = pd.DataFrame({"sl_quantidade_vitimas": victim_quantity}).dropna()
    fig = px.histogram(
        quantity_df, x="sl_quantidade_vitimas", nbins=50,
        title="Distribution of victims per record: full range",
        labels={"sl_quantidade_vitimas": "Victims per record"}
    )
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    save_figure(fig, "quantity_victims_full_range.html")

    p99 = victim_quantity.quantile(.99)
    zoom = quantity_df[quantity_df["sl_quantidade_vitimas"] <= p99]
    fig_zoom = px.histogram(
        zoom, x="sl_quantidade_vitimas", nbins=min(40, max(10, int(zoom["sl_quantidade_vitimas"].nunique()))),
        title="Distribution of victims per record: values up to the 99th percentile",
        labels={"sl_quantidade_vitimas": "Victims per record"}
    )
    fig_zoom.update_layout(template=PLOTLY_TEMPLATE)
    fig_zoom.show()
    save_figure(fig_zoom, "quantity_victims_p99_zoom.html")

    quantity_summary.to_csv(TABLE_DIR / "victim_quantity_summary.csv", encoding="utf-8-sig")
    quantity_diagnostics.to_csv(TABLE_DIR / "victim_quantity_diagnostics.csv", index=False, encoding="utf-8-sig")


,value
count,8401.0
mean,3.287704
std,7.467195
min,1.0
1%,1.0
5%,1.0
25%,1.0
50%,1.0
75%,2.0
90%,10.0


,metric,value
0,IQR,1.0
1,IQR upper review fence,3.5
2,Records above fence,963.0
3,99th percentile,50.0


## 4.14 Missingness as a univariate characteristic

Missingness is itself informative in administrative data. This chart identifies variables whose interpretation is most constrained by incomplete recording. It does not imply that missingness is random.


In [64]:
missing_profile = pd.DataFrame({
    "column": df_analysis.columns,
    "missing_n": [int(df_analysis[c].isna().sum()) for c in df_analysis.columns],
    "missing_pct": [round(df_analysis[c].isna().mean()*100, 2) for c in df_analysis.columns]
}).sort_values("missing_pct", ascending=False)

display(missing_profile)
fig = px.bar(
    missing_profile.sort_values("missing_pct"),
    x="missing_pct", y="column", orientation="h", color="missing_pct",
    color_continuous_scale="Reds", height=max(700, 18*len(missing_profile)),
    title="Missingness in the analytical dataset",
    labels={"missing_pct": "Missing (%)"}
)
fig.update_layout(template=PLOTLY_TEMPLATE, coloraxis_showscale=False)
fig.show()
save_figure(fig, "missingness_analytical_dataset.html")
missing_profile.to_csv(TABLE_DIR / "missingness_profile.csv", index=False, encoding="utf-8-sig")


,column,missing_n,missing_pct
57,vinculo_orgao_pj_do_suspeito,8401,100.00
25,deficiencia_relacionada_a_doenca_rara,8214,97.77
22,sl_vitima_naturalizado_municipio,8189,97.48
21,sl_vitima_naturalizado_uf,8141,96.91
43,sl_suspeito_naturalizado_municipio,8131,96.79
20,sl_vitima_naturalidade,8083,96.21
42,sl_suspeito_naturalizado_uf,8046,95.77
41,sl_suspeito_naturalidade,8015,95.41
59,suspeito_especificacao_etnia,7466,88.87
46,deficiencia_relacionada_a_doenca_rara_suspeito,7454,88.73


## 4.15 Rare-category assessment

Rare categories can destabilize later cross-tabulations, statistical tests, and models. They should not be merged solely because they are rare. First verify whether they are substantively distinct, coding errors, or disclosure risks.


In [65]:
rare_rows = []
for column in analysis_variables:
    table = frequency_table(df_analysis, column, include_missing=False)
    rare = table[
        (table["count"] < RARE_CATEGORY_MIN_COUNT)
        | (table["percent_total"] < RARE_CATEGORY_MIN_PERCENT)
    ]
    for _, row in rare.iterrows():
        rare_rows.append({
            "column": column, "category": row["category"],
            "count": int(row["count"]), "percent_total": row["percent_total"]
        })
rare_categories = pd.DataFrame(rare_rows)
rare_summary = rare_categories.groupby("column", as_index=False).agg(
    rare_categories=("category", "size"),
    records_in_rare_categories=("count", "sum")
).sort_values("rare_categories", ascending=False) if not rare_categories.empty else pd.DataFrame()
display(rare_summary)
display(rare_categories.head(100))
rare_categories.to_csv(TABLE_DIR / "rare_categories.csv", index=False, encoding="utf-8-sig")
rare_summary.to_csv(TABLE_DIR / "rare_category_summary.csv", index=False, encoding="utf-8-sig")


,column,rare_categories,records_in_rare_categories
14,municipio,270,2605
13,motivacao,69,660
24,violacao,58,693
20,relacao_vitima_suspeito,26,296
19,pais,20,261
21,suspeito_nacionalidade,16,190
1,cenario_da_violacao,13,186
18,orientacao_sexual_do_suspeito,8,138
15,nacionalidade_da_vitima,7,142
23,uf,7,139


,column,category,count,percent_total
0,canal_de_atendimento,PORTAL,29,0.35
1,canal_de_atendimento,TELEGRAM,14,0.17
2,canal_de_atendimento,CARTA,1,0.01
3,denuncia_emergencial,VÍTIMA EM SANGRAMENTO,5,0.06
4,denuncia_emergencial,N/D,1,0.01
5,denunciante,PESSOA EM SOFRIMENTO PSÍQUICO,6,0.07
6,denunciante,PESSOA COM DEFICIÊNCIA MENTAL,4,0.05
7,denunciante,O PRÓPRIO AGRESSOR,2,0.02
8,cenario_da_violacao,ESTABELECIMENTO DE SAÚDE,39,0.46
9,cenario_da_violacao,AMBIENTE DE LAZER/ESPORTE/ENTRETENIMENTO,37,0.44


## 4.16 Concentration and diversity indicators

For each categorical variable:

- **Top-category share** measures dominance.
- **Herfindahl-Hirschman Index (HHI)** is the sum of squared category shares. Values closer to 1 indicate stronger concentration.
- **Normalized entropy** approaches 1 when categories are more evenly distributed.

These are descriptive indicators, not tests of inequality or social disparity.


In [66]:
diversity_rows = []
for column in analysis_variables:
    counts = df_analysis[column].dropna().value_counts()
    if counts.empty:
        continue
    shares = counts / counts.sum()
    hhi = float((shares**2).sum())
    entropy = float(-(shares * np.log(shares)).sum())
    normalized_entropy = entropy / np.log(len(shares)) if len(shares) > 1 else 0.0
    diversity_rows.append({
        "column": column,
        "valid_n": int(counts.sum()),
        "categories": int(len(counts)),
        "top_category_share_pct": round(shares.iloc[0]*100, 2),
        "hhi": round(hhi, 4),
        "normalized_entropy": round(normalized_entropy, 4)
    })
concentration_profile = pd.DataFrame(diversity_rows).sort_values("top_category_share_pct", ascending=False)
display(concentration_profile)
concentration_profile.to_csv(TABLE_DIR / "concentration_profile.csv", index=False, encoding="utf-8-sig")

fig = px.scatter(
    concentration_profile, x="normalized_entropy", y="top_category_share_pct",
    size="valid_n", color="categories", hover_name="column",
    title="Category concentration and diversity",
    labels={"normalized_entropy": "Normalized entropy", "top_category_share_pct": "Top category share (%)"}
)
fig.update_layout(template=PLOTLY_TEMPLATE)
fig.show()
save_figure(fig, "category_concentration_diversity.html")


,column,valid_n,categories,top_category_share_pct,hhi,normalized_entropy
1,denuncia_emergencial,8401,5,91.01,0.8342,0.2143
16,nacionalidade_da_vitima,8401,10,87.88,0.7811,0.2106
18,doenca_rara_da_vitima,7170,3,86.79,0.7639,0.4249
19,vitima_preso_a,5395,2,86.28,0.7633,0.5768
24,suspeito_nacionalidade,8392,20,86.25,0.7520,0.1982
26,doenca_rara_do_suspeito,6743,3,85.96,0.7518,0.4363
22,orientacao_sexual_do_suspeito,6648,10,84.01,0.7252,0.2335
2,denunciante,8401,6,79.06,0.6535,0.3529
29,natureza_juridica_do_suspeito,8401,6,77.63,0.6235,0.4310
10,pais,8401,29,77.35,0.6127,0.2884


## 4.17 Automated univariate findings register

The register surfaces variables requiring interpretation. It does not replace substantive review.


In [67]:
finding_rows = []
for _, row in univariate_inventory.iterrows():
    if row["missing_pct"] >= 80:
        finding_rows.append({
            "column": row["column"], "finding_type": "very_high_missingness",
            "finding": f"{row['missing_pct']:.2f}% missing",
            "implication": "Restrict interpretation to the observed subset and assess conditional applicability."
        })
    elif row["missing_pct"] >= 40:
        finding_rows.append({
            "column": row["column"], "finding_type": "high_missingness",
            "finding": f"{row['missing_pct']:.2f}% missing",
            "implication": "Report the denominator and evaluate missingness mechanisms before subgroup analysis."
        })
    if pd.notna(row["mode_pct_valid"]) and row["mode_pct_valid"] >= 90:
        finding_rows.append({
            "column": row["column"], "finding_type": "dominant_category",
            "finding": f"Top category represents {row['mode_pct_valid']:.2f}% of valid observations",
            "implication": "Verify whether this reflects the phenomenon, a default response, or collection design."
        })
    if row["unique_valid"] > 100:
        finding_rows.append({
            "column": row["column"], "finding_type": "high_cardinality",
            "finding": f"{int(row['unique_valid'])} valid categories",
            "implication": "Use top-N charts and review grouping, hierarchy, spelling, and disclosure risk."
        })

univariate_findings = pd.DataFrame(finding_rows)
display(univariate_findings)
univariate_findings.to_csv(TABLE_DIR / "univariate_findings_register.csv", index=False, encoding="utf-8-sig")


,column,finding_type,finding,implication
0,deficiencia_do_suspeito,high_missingness,77.13% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.
1,deficiencia_da_vitima,high_missingness,76.72% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.
2,motivacao,high_missingness,65.05% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.
3,raca_cor_do_suspeito,high_missingness,55.61% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.
4,suspeito_preso,high_missingness,51.95% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.
5,faixa_etaria_do_suspeito,high_missingness,45.64% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.
6,raca_cor_da_vitima,high_missingness,40.71% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.
7,municipio,high_cardinality,313 valid categories,"Use top-N charts and review grouping, hierarchy, spelling, and disclosure risk."
8,denuncia_emergencial,dominant_category,Top category represents 91.01% of valid observations,"Verify whether this reflects the phenomenon, a default response, or collection design."


## 4.18 Export run metadata and figure index


In [68]:
figure_index = pd.DataFrame({
    "figure_file": sorted([p.name for p in FIGURE_DIR.glob("*.html")])
})
figure_index.to_csv(OUTPUT_DIR / "figure_index.csv", index=False, encoding="utf-8-sig")
identifier_conflicts.to_csv(TABLE_DIR / "identifier_conflicts.csv", index=False, encoding="utf-8-sig")
population_summary.to_csv(TABLE_DIR / "population_summary.csv", index=False, encoding="utf-8-sig")
readiness.to_csv(TABLE_DIR / "readiness_checks.csv", index=False, encoding="utf-8-sig")

run_metadata = {
    "data_source": str(DATA_SOURCE),
    "counting_mode": COUNTING_MODE,
    "analytical_unit_label": unit_label,
    "source_rows": int(len(df)),
    "analysis_rows": int(len(df_analysis)),
    "identifier": IDENTIFIER,
    "top_n": TOP_N,
    "rare_category_min_count": RARE_CATEGORY_MIN_COUNT,
    "rare_category_min_percent": RARE_CATEGORY_MIN_PERCENT,
    "include_missing_in_charts": INCLUDE_MISSING_IN_CHARTS,
}
(OUTPUT_DIR / "run_metadata.json").write_text(
    __import__("json").dumps(run_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Tables:", TABLE_DIR.resolve())
print("Interactive figures:", FIGURE_DIR.resolve())


Tables: C:\Projetos\eda-disque100-completo\eda\outputs\step_04_univariate_analysis\tables
Interactive figures: C:\Projetos\eda-disque100-completo\eda\outputs\step_04_univariate_analysis\figures


## 4.19 Interpretation checklist

Before moving to bivariate analysis, document:

1. Which counting mode was used and why.
2. Whether `violacao` is a row-level, complaint-level, hierarchical, or multi-label variable.
3. Which variables have more than 40% missingness.
4. Which fields have dominant categories above 90% of valid observations.
5. Which rare categories require retention, aggregation, correction, or suppression.
6. Whether victim and suspect profiles refer to unique people or complaint-linked records.
7. Whether UF and municipality counts require population denominators from IBGE before rate comparison.
8. Whether collective complaints explain extreme values of `sl_quantidade_vitimas`.
9. Which charts should appear in the paper and which belong in supplementary materials.
10. Which findings are descriptive and which can support hypotheses for Step 5.

## Recommended reporting language

- Report both **count** and **percentage**.
- State whether percentages use all observations or valid observations.
- Describe missingness explicitly for each variable.
- Use “recorded complaints” or “reported violations,” not population prevalence.
- Avoid causal language.
- Avoid describing suspect attributes as proven offender characteristics.
- For geography, present raw volume and population-standardized rates separately.

## Next step

**Step 5: Bivariate Analysis** should examine relationships such as violation by victim age, gender, race/color, vulnerable group, victim-suspect relationship, emergency classification, UF, and time. It should use normalized cross-tabulations, effect sizes such as Cramér's V, minimum-cell checks, and multiple-comparison controls where inferential tests are used.
